In [ ]:
!pip install lancedb openai contextgem pyarrow requests

In [ ]:
!pip install lancedb openai contextgem pyarrow requests google-genai numpy

In [3]:
from google.colab import userdata
import os

# Set the key into your environment variables for both OpenAI and ContextGem
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [ ]:
import os
import lancedb
import pyarrow as pa
from openai import OpenAI
import requests
import shutil
from contextgem import Document, DocumentLLM, StringConcept

# 1. Initialize Clients
client = OpenAI(api_key=OPENAI_API_KEY)

# 2. Read movie corpus file
movie_corpus_url = "https://raw.githubusercontent.com/Balachandar-Ganesan/GenAIArchitect/refs/heads/main/rajinikanth_movies_list.txt"
response = requests.get(movie_corpus_url)
response.raise_for_status()
movie_corpus = response.text

# 3. Process with distinct, high-recall extraction concepts
doc = Document(raw_text=movie_corpus)
llm = DocumentLLM(model="openai/gpt-4o", api_key=OPENAI_API_KEY)

# Pass 1: Focus specifically on Multiple Characters (Physically separate entities)
role_concept = StringConcept(
    name="Multiple Roles",
    description="Identify EVERY movie where Rajinikanth plays 2 or more distinct physical characters. This includes: 1) Twins/Triple roles (Moondru Mugam), 2) Father/Son (Netrikkan), 3) Robot/Human (Enthiran), and 4) Lookalikes (Billa). Ensure Kochadaiiyaan (Rana/Kochadaiiyaan) is included.",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

# Pass 2: Focus specifically on Identity Deception (Faking an identity/disguise)
deception_concept = StringConcept(
    name="Identity Deception",
    description="Identify EVERY movie where a character uses a disguise, fake identity, or impersonates another. Note: Billa (impersonating a lookalike), Thillu Mullu (fake twin), and Baasha (hidden past identity).",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

doc.add_concepts([role_concept, deception_concept])
doc = llm.extract_all(doc, overwrite_existing=True)

# 4. Reset and Re-index LanceDB
db_path = "./movie_rag_db"
if os.path.exists(db_path): shutil.rmtree(db_path)

db = lancedb.connect(db_path)
schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), 1536)),
    pa.field("text", pa.string()),
    pa.field("justification", pa.string()),
    pa.field("category", pa.string())
])

def get_embedding(text):
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

data_to_insert = []
for c_name in ["Multiple Roles", "Identity Deception"]:
    concept = doc.get_concept_by_name(c_name)
    if concept and concept.extracted_items:
        for item in concept.extracted_items:
            content = f"Category: {c_name} | Movie: {item.value} | Details: {item.justification}"
            data_to_insert.append({
                "vector": get_embedding(content),
                "text": str(item.value),
                "justification": str(item.justification),
                "category": c_name
            })

if data_to_insert:
    table = db.create_table("movie_plots", schema=schema, mode="overwrite")
    table.add(data_to_insert)
    print(f"Success: Indexed {len(data_to_insert)} entries.")

# 5. RAG Engine with categorical separation
def ask_movie_rag(query_str, category_filter):
    query_vector = get_embedding(query_str)
    # Increase limit to ensure all extracted movies are captured
    search_results = table.search(query_vector).where(f"category = '{category_filter}'").limit(25).to_list()

    context = "\n".join([f"[{res['category']}] {res['text']}: {res['justification']}" for res in search_results])

    system_prompt = (
        "Use ONLY the provided context from the movie list text. "
        "List all applicable movies clearly. For overlapping cases like 'Billa', "
        "explicitly state that it fits both categories because it has physically distinct characters "
        "AND one character impersonates the other."
    )

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {query_str}"}
        ]
    )
    return response.choices[0].message.content

# 6. Final Validation
print("\n--- Logic-Validated Movie Analysis ---\n")
q1 = "List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan)."
q2 = "List every movie involving identity deception, disguises, or faked identities (e.g. Billa, Thillu Mullu, Baasha)."

print(f"Q1: {q1}\nA: {ask_movie_rag(q1, 'Multiple Roles')}\n")
print(f"Q2: {q2}\nA: {ask_movie_rag(q2, 'Identity Deception')}")

In [8]:
!pip install lancedb
import os
import shutil
import requests
import pyarrow as pa
import lancedb
import google.generativeai as genai # Reverting to google.generativeai
from google.generativeai import types # Reverting to google.generativeai.types
from contextgem import Document, DocumentLLM, StringConcept

# 1. Initialize Gemini Client
# Assumes you have configured GEMINI_API_KEY in your environment variables
#GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY) # Try configure again with google.generativeai

# --- Diagnostic Step: List available models --- START
print("Listing available Gemini models:")
for m in genai.list_models(): # Should work with google.generativeai
    # Check if the model supports text embedding
    if "embedContent" in m.supported_generation_methods:
        print(f"  Embedding Model: {m.name} (Max input tokens: {m.input_token_limit})")
    # Check if the model supports content generation
    if "generateContent" in m.supported_generation_methods:
        print(f"  Generative Model: {m.name} (Max input tokens: {m.input_token_limit})")

# --- Diagnostic Step: List available models --- END

# 2. Read movie corpus file
movie_corpus_url = "https://raw.githubusercontent.com/Balachandar-Ganesan/GenAIArchitect/refs/heads/main/rajinikanth_movies_list.txt"
response = requests.get(movie_corpus_url)
response.raise_for_status()
movie_corpus = response.text

# 3. Process with distinct, high-recall extraction concepts

doc = Document(raw_text=movie_corpus)
# Swapped the processing LLM backend to Gemini
llm = DocumentLLM(
    model="gemini/gemini-3.6-flash", # contextgem might handle this model name internally
    api_key=GEMINI_API_KEY
)

# Pass 1: Focus specifically on Multiple Characters (Physically separate entities)
role_concept = StringConcept(
    name="Multiple Roles",
    description="Identify EVERY movie where Rajinikanth plays 2 or more distinct physical characters. This includes: 1) Twins/Triple roles (Moondru Mugam), 2) Father/Son (Netrikkan), 3) Robot/Human (Enthiran), and 4) Lookalikes (Billa). Ensure Kochadaiiyaan (Rana/Kochadaiiyaan) is included.",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

# Pass 2: Focus specifically on Identity Deception (Faking an identity/disguise)
deception_concept = StringConcept(
    name="Identity Deception",
    description="Identify EVERY movie where a character uses a disguise, fake identity, or impersonates another. Note: Billa (impersonating a lookalike), Thillu Mullu (fake twin), and Baasha (hidden past identity).",
    add_references=True,
    reference_depth="sentences",
    add_justifications=True,
    justification_depth="comprehensive"
)

doc.add_concepts([role_concept, deception_concept])
doc = llm.extract_all(doc, overwrite_existing=True)

# 4. Reset and Re-index LanceDB
db_path = "./movie_rag_db"
if os.path.exists(db_path):
    shutil.rmtree(db_path)

db = lancedb.connect(db_path)

# CRITICAL CRITERIA UPDATE: gemini-embedding-001 dimensions length is 3072
schema = pa.schema([
    pa.field("vector", pa.list_(pa.float32(), 3072)), # Updated dimension from 768 to 3072
    pa.field("text", pa.string()),
    pa.field("justification", pa.string()),
    pa.field("category", pa.string())
])

def get_embedding(text):
    try:
        # Use genai.embed_content with google.generativeai
        response = genai.embed_content(
            model="models/gemini-embedding-001", # Use full model path for google.generativeai, changed to listed model
            content=text, # Argument name is 'content'
            task_type="SEMANTIC_SIMILARITY" # Use string for task_type with google.generativeai
        )
        return response['embedding']
    except Exception as e:
        print(f"Error generating text embedding: {e}")
        return None

data_to_insert = []
for c_name in ["Multiple Roles", "Identity Deception"]:
    concept = doc.get_concept_by_name(c_name)
    if concept and concept.extracted_items:
        for item in concept.extracted_items:
            content = f"Category: {c_name} | Movie: {item.value} | Details: {item.justification}"
            data_to_insert.append({
                "vector": get_embedding(content),
                "text": str(item.value),
                "justification": str(item.justification),
                "category": c_name
            })

if data_to_insert:
    table = db.create_table("movie_plots", schema=schema, mode="overwrite")
    table.add(data_to_insert)
    print(f"Success: Indexed {len(data_to_insert)} entries.")

# 5. RAG Engine with categorical separation
def ask_movie_rag(query_str, category_filter):
    query_vector = get_embedding(query_str)
    # Search LanceDB local records matrix
    search_results = table.search(query_vector).where(f"category = '{category_filter}'").limit(25).to_list()

    context = "\n".join([f"[{res['category']}] {res['text']}: {res['justification']}" for res in search_results])

    system_prompt = (
        "Use ONLY the provided context from the movie list text. "
        "List all applicable movies clearly. For overlapping cases like 'Billa', "
        "explicitly state that it fits both categories because it has physically distinct characters "
        "AND one character impersonates the other."
    )

    # Combined System prompt context instruction block for native text generation syntax
    rag_prompt = f"""
    {system_prompt}

    CONTEXT:
    {context}

    QUESTION: {query_str}
    """

    # Use genai.GenerativeModel for generation with google.generativeai
    generative_model = genai.GenerativeModel(
        "models/gemini-3.6-flash" # Changed to models/gemini-3.6-flash as 2.5-flash is deprecated
    )
    response = generative_model.generate_content(contents=[rag_prompt])
    return response.text

# 6. Final Validation
print("\n--- Logic-Validated Movie Analysis ---\n")
q1 = "List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan)."
q2 = "List every movie involving identity deception, disguises, or faked identities (e.g. Billa, Thillu Mullu, Baasha)."

print(f"Q1: {q1}\nA: {ask_movie_rag(q1, 'Multiple Roles')}\n")
print(f"Q2: {q2}\nA: {ask_movie_rag(q2, 'Identity Deception')}")

Listing available Gemini models:
  Generative Model: models/gemini-2.5-flash (Max input tokens: 1048576)
  Generative Model: models/gemini-2.5-pro (Max input tokens: 1048576)
  Generative Model: models/gemini-2.5-flash-preview-tts (Max input tokens: 8192)
  Generative Model: models/gemini-2.5-pro-preview-tts (Max input tokens: 8192)
  Generative Model: models/gemma-4-26b-a4b-it (Max input tokens: 262144)
  Generative Model: models/gemma-4-31b-it (Max input tokens: 262144)
  Generative Model: models/gemini-flash-latest (Max input tokens: 1048576)
  Generative Model: models/gemini-flash-lite-latest (Max input tokens: 1048576)
  Generative Model: models/gemini-pro-latest (Max input tokens: 1048576)
  Generative Model: models/gemini-2.5-flash-lite (Max input tokens: 1048576)
  Generative Model: models/gemini-2.5-flash-image (Max input tokens: 32768)
  Generative Model: models/gemini-3-flash-preview (Max input tokens: 1048576)
  Generative Model: models/gemini-3.1-pro-preview (Max input tok

/usr/local/lib/python3.13/dist-packages/litellm/litellm_core_utils/logging_worker.py:75: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._queue = None


Success: Indexed 10 entries.

--- Logic-Validated Movie Analysis ---

Q1: List every movie where Rajinikanth plays MULTIPLE physically distinct characters (e.g. Billa, Enthiran, Moondru Mugam, Kochadaiiyaan).
A: Based on the provided context, here are all the movies where Rajinikanth plays multiple physically distinct characters:

1. **Kochadaiiyaan**: Rajinikanth plays multiple physical roles, including Commander Kochadaiiyaan and his sons, Rana and Sena.
2. **Enthiran**: Rajinikanth plays two distinct physical characters: the scientist Dr. Vaseegaran and the humanoid robot Chitti.
3. **Billa**: Rajinikanth plays two distinct physical characters: the underworld don Billa and his village lookalike Rajappa. *(Note: This fits both categories because it has physically distinct characters AND one character impersonates the other.)*
4. **Moondru Mugam**: Rajinikanth plays three distinct physical characters: the police officer Alex Pandian and his twin sons, Arun and John.
5. **Netrikkan**: 